> This notebook is still WIP (Work in Progress)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import json
import pathlib
import time
import os

In [ ]:
# ----------------- Configuration -----------------
BATCH_SIZE = 32
BLOCK_SIZE = 128
MAX_ITERS = 5000
EVAL_INTERVAL = 250  # Evaluate more frequently to save checkpoints
LEARNING_RATE = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_ITERS = 200
N_EMBD = 384
N_HEAD = 6
N_LAYER = 6
DROPOUT = 0.2

In [ ]:
# --- Paths ---
PREPARED_DATA_DIR = "/kaggle/input/sanskrit-text-corpus"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"  # Directory to save checkpoints
TRAIN_DATA_PATH = pathlib.Path(PREPARED_DATA_DIR) / "train.txt"
META_PATH = pathlib.Path(PREPARED_DATA_DIR) / "meta.json"
# ----------------------------------------------------

In [ ]:
print(f"Using device: {DEVICE}")
torch.manual_seed(1337)

# Create checkpoint directory if it doesn't exist
pathlib.Path(CHECKPOINT_DIR).mkdir(exist_ok=True)

In [ ]:
# 1. --- Data Loading (Same as before) ---
with open(META_PATH, "r") as f:
    meta = json.load(f)
vocab_size = meta["vocab_size"]
stoi = meta["stoi"]
itos = meta["itos"]
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[str(i)] for i in l])

with open(TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    data = torch.tensor(encode(f.read()), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i : i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1 : i + BLOCK_SIZE + 1] for i in ix])
    x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[Block(N_EMBD, n_head=N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
model = GPTLanguageModel()
m = model.to(DEVICE)
print(f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# --- CHECKPOINT LOADING ---
start_iter = 0
best_val_loss = float("inf")
ckpt_path = pathlib.Path(CHECKPOINT_DIR) / "ckpt.pt"
if os.path.exists(ckpt_path):
    print("Resuming training from checkpoint.")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_iter = checkpoint["iter"]
    best_val_loss = checkpoint["best_val_loss"]
    print(
        f"Resumed from iteration {start_iter} with best validation loss {best_val_loss:.4f}"
    )

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
print("\n--- Starting Training ---")
start_time = time.time()
for iter in range(start_iter, MAX_ITERS):
    # Sample a batch of data
    xb, yb = get_batch("train")

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Every once in a while, evaluate and save checkpoint
    if iter % EVAL_INTERVAL == 0 or iter == MAX_ITERS - 1:
        losses = estimate_loss()
        current_time = time.time()
        elapsed = current_time - start_time
        print(
            f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, elapsed: {elapsed:.2f}s"
        )

        # --- CHECKPOINT SAVING ---
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            print(
                f"New best validation loss: {best_val_loss:.4f}. Saving checkpoint..."
            )
            checkpoint = {
                "iter": iter,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
            }
            torch.save(checkpoint, ckpt_path)

print("--- Training Complete ---")

In [ ]:
# todo: create seperate script later
print("\n--- Generating Sample Text from Final Model ---")
context = torch.tensor([encode("\n")], dtype=torch.long, device=DEVICE)
generated_chars = decode(m.generate(context, max_new_tokens=200)[0].tolist())
print(generated_chars)
print("-----------------------\n")